# From Rasters to Polygons and Back

This notebook goes from a raster to polygons and back, with the Copernicus crop type
map and Sentinel-2 data of the Copernicus Data Space Ecosystem (CDSE).

In this notebook, we will:

1. Load the crop type map of the Copernicus Land Monitoring Service (CLMS).
2. Make one polygon for each crop zone with `raster_to_vector`.
3. Keep the crop zones that touch our fields with `filter_vector`.
4. Compute the mean NDVI of each crop zone with `aggregate_spatial`.
5. Put the crop zones on the Sentinel-2 grid with `vector_to_raster`.
6. Cut the NDVI to the fields with `filter_spatial`.
7. Smooth the NDVI inside each field with `apply_polygon`.

See [Vector Data Cubes](https://sentinel-hub.github.io/titiler-openeo/vector-cube/) for the processes and formats.

## Requirements

- A CDSE account, for the login.
- Python packages: `openeo>=0.54`, `geopandas`, `pyarrow`, `rasterio` and
  `matplotlib`. In this repository: `uv sync --group notebook --extra server`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rasterio.plot
from matplotlib.patches import Patch

## Connect to the back-end

In [ ]:
import os

import openeo

# The deployment with the vector data cubes. Set TITILER_OPENEO_URL to use
# another one, for example a local server: http://localhost:8082
URL = os.environ.get("TITILER_OPENEO_URL", "https://openeo.ds.io")

# Login with your Copernicus Data Space Ecosystem account (device flow):
# open the link that is printed below and enter the code.
connection = openeo.connect(URL).authenticate_oidc_device()

In [ ]:
processes = {process["id"] for process in connection.list_processes()}
formats = connection.list_file_formats()
missing = sorted(
    {
        "raster_to_vector",
        "vector_to_raster",
        "filter_vector",
        "filter_spatial",
        "apply_polygon",
    }
    - processes
)
if (
    missing
    or "GeoParquet" not in formats["output"]
    or "FlatGeobuf" not in formats["input"]
):
    raise RuntimeError(
        f"{URL} does not have the vector data cube features yet "
        f"(missing processes: {missing})."
    )
print(f"✅ {URL} supports vector data cubes.")
print(
    "Vector input formats:",
    sorted(k for k, v in formats["input"].items() if "vector" in v["gis_data_types"]),
)

In [ ]:
# The demo fields, as GeoJSON, FlatGeobuf and GeoParquet files. The back-end
# downloads them, so the URL must be public. For a local server, serve the
# folder docs/src/notebooks/data and set DEMO_DATA_URL=http://localhost:8000
DEMO_DATA_URL = os.environ.get(
    "DEMO_DATA_URL",
    "https://raw.githubusercontent.com/sentinel-hub/titiler-openeo/main/docs/src/notebooks/data",
)

# The downloaded results go to this folder.
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

# The area of the fields, near Mol (Belgium).
BBOX = {"west": 5.05, "south": 51.21, "east": 5.09, "north": 51.24}

fields = connection.load_url(f"{DEMO_DATA_URL}/mol_fields.fgb", format="FlatGeobuf")
fields.download(OUT / "fields.geojson", format="GeoJSON")
fields_gdf = gpd.read_file(OUT / "fields.geojson")

## 1. The crop type map

The CLMS crop type layer (`clms_vlcc_crop-types_europe_10m_yearly_v1`) gives the main
crop of each 10 m pixel in Europe, once a year. It is in the equal-area projection
EPSG:3035.

The back-end loads 1024 pixels over the extent by default (about 3 m here). We use
`resample_spatial` to go back to 10 m, with the `near` method that keeps the class
codes.

The legend is Table 0-1 of the
[CLMS Croplands Product User Manual](https://land.copernicus.eu/en/technical-library/product-user-manual-croplands-2017-present/@@download/file)
(v2.6), with its colours.

In [ ]:
# Code: (name, colour) from the CLMS Croplands Product User Manual v2.6, Table 0-1.
CROP_TYPES = {
    0: ("No cropland", (240, 240, 240)),
    1110: ("Wheat", (238, 110, 50)),
    1120: ("Barley", (251, 162, 74)),
    1130: ("Maize", (250, 220, 20)),
    1140: ("Rice", (233, 67, 1)),
    1150: ("Other cereals", (232, 169, 149)),
    1210: ("Fresh vegetables", (174, 199, 232)),
    1220: ("Dry pulses", (72, 151, 191)),
    1310: ("Potatoes", (201, 140, 67)),
    1320: ("Sugar beet", (156, 91, 12)),
    1410: ("Sunflower", (255, 121, 121)),
    1420: ("Soybeans", (168, 106, 150)),
    1430: ("Rapeseed", (227, 119, 194)),
    1440: ("Flax, cotton and hemp", (247, 182, 210)),
    2100: ("Grapes", (219, 219, 141)),
    2200: ("Olives", (193, 206, 18)),
    2310: ("Fruits", (121, 160, 58)),
    2320: ("Nuts", (90, 124, 48)),
    3100: ("Unclassified arable crop", (215, 215, 215)),
    3200: ("Unclassified permanent crop", (171, 171, 171)),
}


def crop_colour(code):
    """Return the CLMS colour of a crop code, as matplotlib RGB."""
    return tuple(c / 255 for c in CROP_TYPES.get(int(code), ("Unknown", (0, 0, 0)))[1])


def crop_legend(ax, codes):
    """Add a legend with the names of the crop codes."""
    handles = [
        Patch(color=crop_colour(c), label=f"{c} {CROP_TYPES[c][0]}")
        for c in sorted(codes)
        if c in CROP_TYPES
    ]
    ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1, 0.5))


def show_crops(ax, array, extent):
    """Show a raster of crop codes with the CLMS colours."""
    rgb = np.ones(array.shape + (3,))
    for code in np.unique(array.compressed()):
        rgb[array.filled(65535) == code] = crop_colour(code)
    ax.imshow(rgb, extent=extent)
    crop_legend(ax, set(np.unique(array.compressed()).tolist()))


crops = connection.load_collection(
    "clms_vlcc_crop-types_europe_10m_yearly_v1",
    spatial_extent=BBOX,
    temporal_extent=["2024-01-01", "2025-01-01"],
    bands=["data"],
).resample_spatial(resolution=10, method="near")

crops.download(OUT / "crop_types.tif", format="GTiff")
with rasterio.open(OUT / "crop_types.tif") as dataset:
    crop_array = dataset.read(1, masked=True)
    crop_extent = rasterio.plot.plotting_extent(dataset)
    print(
        f"{dataset.crs}, {dataset.width} x {dataset.height} pixels of {dataset.res[0]:.0f} m"
    )

fig, ax = plt.subplots(figsize=(8, 6))
show_crops(ax, crop_array, crop_extent)
ax.set_title("CLMS crop types 2024")
plt.show()

## 2. Polygons with `raster_to_vector`

`raster_to_vector` (alias `vectorize`) makes one polygon for each zone of 4-connected
pixels with the same value. The value of each polygon is the crop code. The polygons
are in the CRS of the raster: EPSG:3035 is an equal-area CRS, so the areas are correct.

The back-end limits the number of polygons and the work of the polygonization
(`OutputLimitExceeded`). A crop map is smooth, so it is far below the limits.

In [ ]:
zones = crops.raster_to_vector()
zones.download(OUT / "crop_zones.gpkg", format="GPKG")

zones_gdf = gpd.read_file(OUT / "crop_zones.gpkg")
zones_gdf["crop"] = [CROP_TYPES[code][0] for code in zones_gdf["value"]]
print(f"{len(zones_gdf)} crop zones, CRS {zones_gdf.crs}")

areas = (
    zones_gdf.assign(area_ha=zones_gdf.area / 10_000)
    .groupby("crop")["area_ha"]
    .agg(["count", "sum"])
)
print(
    areas.rename(columns={"count": "zones", "sum": "area (ha)"})
    .round(1)
    .sort_values("area (ha)", ascending=False)
)

ax = zones_gdf.plot(
    color=[crop_colour(c) for c in zones_gdf["value"]],
    edgecolor="white",
    linewidth=0.3,
    figsize=(8, 6),
)
fields_gdf.to_crs(zones_gdf.crs).boundary.plot(ax=ax, color="black")
crop_legend(ax, set(zones_gdf["value"]))
ax.set_title("Crop zones (white edges) and the demo fields (black)")
plt.show()

## 3. The crop zones of our fields: `filter_vector`

`filter_vector` keeps the crop zones that intersect the demo fields. The fields are
reprojected to the CRS of the crop zones by the back-end.

In [ ]:
near = zones.filter_vector(geometries=fields, relation="intersects")
near.download(OUT / "crop_zones_near.geojson", format="GeoJSON")

near_gdf = gpd.read_file(OUT / "crop_zones_near.geojson")
print(f"{len(near_gdf)} crop zones intersect the fields:")
print(
    near_gdf["value"].map(lambda code: CROP_TYPES[code][0]).value_counts().to_string()
)

The polygons are also ordinary vector data on the client. With geopandas, we compute
the share of each crop in each field (in %).

In [ ]:
fields_3035 = fields_gdf.to_crs(zones_gdf.crs)[["name", "geometry"]]
share = gpd.overlay(fields_3035, zones_gdf[["crop", "geometry"]], how="intersection")
share["area"] = share.area
table = share.pivot_table(
    index="name", columns="crop", values="area", aggfunc="sum", fill_value=0
)
(100 * table.div(table.sum(axis=1), axis=0)).round(0)

## 4. Zonal statistics on the polygons

The polygons of `raster_to_vector` are a vector data cube, so `aggregate_spatial` can
use them directly. We compute the mean NDVI of each crop zone on a clear day
(19 August 2024).

The crop code of each zone stays in the property `value`. The NDVI mean has the same
column name, so the GeoPackage writer renames it to `value_1`.

In [ ]:
s2 = connection.load_collection(
    "sentinel-2-l2a",
    spatial_extent=BBOX,
    temporal_extent=["2024-08-19", "2024-08-20"],
    bands=["B04_10m", "B08_10m"],
)
ndvi = s2.ndvi(nir="B08_10m", red="B04_10m")

stats = ndvi.aggregate_spatial(geometries=near, reducer="mean")
stats.download(OUT / "crop_zone_ndvi.gpkg", format="GPKG")

stats_gdf = gpd.read_file(OUT / "crop_zone_ndvi.gpkg").rename(
    columns={"value": "crop_code", "value_1": "ndvi"}
)
stats_gdf["crop"] = [CROP_TYPES[code][0] for code in stats_gdf["crop_code"]]
stats_gdf.groupby("crop")["ndvi"].agg(["count", "mean"]).round(3)

## 5. Back to a raster: `vector_to_raster`

`vector_to_raster` writes the values of a vector data cube on the grid of a target
raster: here the NDVI (UTM, 1024 pixels). A pixel gets the value of the polygon that
contains its center. The pixels of the target are not read.

In [ ]:
crop_grid = zones.vector_to_raster(ndvi)
crop_grid.download(OUT / "crop_grid.tif", format="GTiff")
ndvi.download(OUT / "ndvi.tif", format="GTiff")

with (
    rasterio.open(OUT / "crop_grid.tif") as grid,
    rasterio.open(OUT / "ndvi.tif") as image,
):
    print(
        f"crop grid: {grid.crs}, {grid.width} x {grid.height} | NDVI: {image.crs}, {image.width} x {image.height}"
    )
    grid_array, grid_extent = (
        grid.read(1, masked=True),
        rasterio.plot.plotting_extent(grid),
    )
    ndvi_array, ndvi_extent = (
        image.read(1, masked=True),
        rasterio.plot.plotting_extent(image),
    )

fig, (left, right) = plt.subplots(1, 2, figsize=(16, 6))
show_crops(left, grid_array, grid_extent)
left.set_title("Crop zones on the Sentinel-2 grid")
right.imshow(ndvi_array, cmap="YlGn", vmin=0, vmax=1, extent=ndvi_extent)
right.set_title("NDVI, 19 August 2024")
plt.tight_layout()
plt.show()

## 6. Cut the raster to the fields: `filter_spatial`

`filter_spatial` keeps the pixels whose center is in a field. The raster is cropped
to the smallest window around the fields, and the other pixels of the window have no
data.

In [ ]:
in_fields = ndvi.filter_spatial(geometries=fields)
in_fields.download(OUT / "ndvi_fields.tif", format="GTiff")

with rasterio.open(OUT / "ndvi_fields.tif") as dataset:
    print(f"{dataset.width} x {dataset.height} pixels (from 1024 x 1024)")
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(
        dataset.read(1, masked=True),
        cmap="YlGn",
        vmin=0,
        vmax=1,
        extent=rasterio.plot.plotting_extent(dataset),
    )
    fields_gdf.to_crs(dataset.crs).boundary.plot(ax=ax, color="black")
    ax.set_title("NDVI of the fields")
    plt.show()

## 7. Process each field separately: `apply_polygon`

`apply_polygon` runs a process once for each field, on a sub-cube cropped to the
field. The pixels outside the field have no data in the sub-cube, and the pixels
outside all fields get `mask_value` in the result.

Here the process smooths the NDVI with the mean of the 5 x 5 neighbours **in the same
field**. `apply_kernel` replaces no-data with 0 (openEO specification), so a plain
`apply_kernel` would lower the values at the field edge. The process divides the sum
of the NDVI by the number of field pixels in the window instead (a normalized
convolution):

- `total`: the sum of the NDVI of the neighbours (no-data counts as 0);
- `count`: the same sum on a cube of 1 inside the field;
- `total / count`, with `merge_cubes` and the `divide` resolver.

In [ ]:
KERNEL = [[1.0] * 5] * 5


def field_mean(data):
    """Mean NDVI of the neighbours in the same field."""
    total = data.apply_kernel(kernel=KERNEL)
    count = data.apply(lambda x: x * 0 + 1).apply_kernel(kernel=KERNEL)
    return total.merge_cubes(count, overlap_resolver="divide")


smooth = ndvi.apply_polygon(geometries=fields, process=field_mean)
smooth.download(OUT / "ndvi_field_mean.tif", format="GTiff")

# The same 5 x 5 mean on the whole image, for comparison.
everywhere = ndvi.apply_kernel(kernel=[[1 / 25] * 5] * 5)
everywhere.download(OUT / "ndvi_mean.tif", format="GTiff")

Inside a field, the two means are equal. At the field edges, the whole-image mean also
uses the pixels of the road or the next field; the `apply_polygon` mean does not. The
figure zooms on field B.

In [ ]:
with (
    rasterio.open(OUT / "ndvi_field_mean.tif") as a,
    rasterio.open(OUT / "ndvi_mean.tif") as b,
):
    in_field, whole = a.read(1, masked=True), b.read(1, masked=True)
    transform, crs = a.transform, a.crs

difference = np.ma.abs(in_field - whole)
print(f"Field pixels: {in_field.count()}, equal means: {(difference < 1e-6).sum()}")

# Zoom on field B, with a margin of 50 m.
west, south, east, north = (
    fields_gdf.to_crs(crs).loc[fields_gdf["name"] == "field B"].total_bounds
)
col0, row0 = ~transform * (west - 50, north + 50)
col1, row1 = ~transform * (east + 50, south - 50)
rows, cols = slice(int(row0), int(row1) + 1), slice(int(col0), int(col1) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, array, title, cmap, vmax in [
    (axes[0], ndvi_array, "NDVI", "YlGn", 1),
    (axes[1], in_field, "Mean in the field (apply_polygon)", "YlGn", 1),
    (axes[2], difference, "Difference to the whole-image mean", "Reds", 0.1),
]:
    image = ax.imshow(array[rows, cols], cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(title)
    ax.set_axis_off()
fig.colorbar(image, ax=axes[2], shrink=0.8)
plt.show()

## Interpretation

- The crop zones of `raster_to_vector` are real vector data: they have exact areas in
  the equal-area CRS, and other processes use them directly (`filter_vector`,
  `aggregate_spatial`, `vector_to_raster`).
- Maize covers most of the cropland around the fields in 2024, and its mean NDVI on
  19 August is high (above 0.7).
- Fields C and D are mostly maize in the CLMS map, and fields A and B are mostly "No
  cropland". Maize grows late: this agrees with the late NDVI peak of fields C and D
  in the [field monitoring notebook](https://sentinel-hub.github.io/titiler-openeo/notebooks/vector_field_monitoring/).
- `apply_polygon` runs a process on each field without the pixels around the field.
  The difference to the whole-image mean is only at the field edges.

## Limitations

- The requests are synchronous: keep the area small.
- The default grid has 1024 pixels over the extent (see [Concepts](https://sentinel-hub.github.io/titiler-openeo/concepts/)).
  `resample_spatial` gives the 10 m grid back, but not exactly the grid of the source.
- `apply_kernel` replaces no-data with `replace_invalid` (0 by default): use a
  normalized convolution, as above, for a mean inside a polygon.
- The polygons of `raster_to_vector` follow the pixel edges.
- The other known limits are in [Vector Data Cubes](https://sentinel-hub.github.io/titiler-openeo/vector-cube/#known-limits).

## References

- [Vector Data Cubes](https://sentinel-hub.github.io/titiler-openeo/vector-cube/).
- openEO processes: [filter_spatial](https://processes.openeo.org/#filter_spatial),
  [apply_polygon](https://processes.openeo.org/#apply_polygon),
  [filter_vector](https://processes.openeo.org/#filter_vector).
- Crop types: Copernicus Land Monitoring Service, HRL Vegetated Land Cover
  Characteristics, Crop Types 2024. See the
  [Croplands Product User Manual](https://land.copernicus.eu/en/technical-library/product-user-manual-croplands-2017-present/@@download/file).
- Contains modified Copernicus Sentinel data (2024), from the
  [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/).